# Test locale API PoliMillionaire

Notebook minimo per verificare che il client API estratto localmente funzioni: import, login, lista competizioni, leaderboard e smoke test opzionale su una partita.

Nota: la password e salvata in chiaro nel notebook per comodita di test locale. Non condividere questo file pubblicamente senza rimuoverla.

In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        client_dir = candidate / "api_client" / "NLP_assignment_api_client" / "millionaire_client"
        if client_dir.exists():
            return candidate
    raise FileNotFoundError("Could not find api_client/NLP_assignment_api_client/millionaire_client")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
CLIENT_PARENT = PROJECT_ROOT / "api_client" / "NLP_assignment_api_client"

if str(CLIENT_PARENT) not in sys.path:
    sys.path.insert(0, str(CLIENT_PARENT))

print("Project root:", PROJECT_ROOT)
print("Client path:", CLIENT_PARENT)

In [ ]:
from millionaire_client import (
    MillionaireClient,
    AuthenticationError,
    MillionaireError,
    TimeoutError,
)

API_URL = "http://131.175.15.22:51111/"
USERNAME = "NeuroniNegroni"
PASSWORD = "TomGiuliaGio"

client = MillionaireClient(API_URL, timeout=30)

try:
    user = client.login(USERNAME, PASSWORD)
    print(f"Login OK: {user.username} (role: {user.role})")
except AuthenticationError as exc:
    print("Login failed:", exc)
    raise

## Competizioni disponibili

In [ ]:
competitions = client.competitions.list_all()

for comp in competitions:
    print(f"{comp.id}: {comp.name} | max levels: {comp.max_levels} | questions: {comp.question_count}")

## Leaderboard di una competizione

Modifica `COMPETITION_ID` se vuoi controllare una competizione diversa.

In [ ]:
COMPETITION_ID = competitions[0].id if competitions else 1

leaderboard = client.leaderboard.get(competition_id=COMPETITION_ID, limit=10)
print(f"Leaderboard: {leaderboard.competition.name}")

for i, entry in enumerate(leaderboard.entries, 1):
    marker = " <- you" if entry.username == USERNAME else ""
    print(f"{i}. {entry.username}: score={entry.score}, level={entry.reached_level}{marker}")

## Smoke test opzionale: avvio partita

Questa cella avvia una partita vera e fa partire il timer della prima domanda. Lasciala disattivata finche non vuoi testare anche `client.game.start()`.

In [ ]:
RUN_GAME_SMOKE_TEST = False

if RUN_GAME_SMOKE_TEST:
    game = client.game.start(competition_id=COMPETITION_ID)
    question = game.current_question

    print("Session ID:", game.session_id)
    print("Current level:", game.current_level)
    print("Time remaining:", game.time_remaining)
    print("Question:", question.text if question else None)

    if question:
        for opt in question.options:
            print(f"{opt.id}: {opt.text}")
else:
    print("Smoke test partita disattivato. Imposta RUN_GAME_SMOKE_TEST = True per avviare una partita.")

## Funzione base per il prossimo step

Questa e la forma minima della funzione che poi sostituiremo con random baseline, LLM locale, RAG, tool calculator, ecc.

In [ ]:
def choose_first_option(question):
    return question.options[0].id


def print_question(question):
    print(question.text)
    for opt in question.options:
        print(f"{opt.id}: {opt.text}")